In [3]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


In [4]:
df = pd.read_csv("../data/heart.csv")

df['Heart Disease'] = df['Heart Disease'].map({
    'Absence': 0,
    'Presence': 1
})

X = df.drop('Heart Disease', axis=1)
y = df['Heart Disease']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

lr = LogisticRegression(max_iter=1000)
dt = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(random_state=42)

lr.fit(X_train, y_train)
dt.fit(X_train, y_train)
rf.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
y_pred_dt = dt.predict(X_test)
y_pred_rf = rf.predict(X_test)


In [5]:
def evaluate_model(y_test, y_pred):
    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred)
    }


In [6]:
results = pd.DataFrame({
    "Logistic Regression": evaluate_model(y_test, y_pred_lr),
    "Decision Tree": evaluate_model(y_test, y_pred_dt),
    "Random Forest": evaluate_model(y_test, y_pred_rf)
})

results


,Logistic Regression,Decision Tree,Random Forest
Accuracy,0.907407,0.685185,0.796296
Precision,0.900000,0.576923,0.777778
Recall,0.857143,0.714286,0.666667
F1 Score,0.878049,0.638298,0.717949


In [7]:
confusion_matrix(y_test, y_pred_lr)
confusion_matrix(y_test, y_pred_dt)
confusion_matrix(y_test, y_pred_rf)


array([[29,  4],
       [ 7, 14]], dtype=int64)

Among the three models evaluated, Logistic Regression achieved the highest accuracy, precision, recall, and F1-score. Since recall is a critical metric in medical risk prediction to minimize false negatives, Logistic Regression was selected as the final model for this project.

In [10]:
# Probability of class 1 (Presence)
y_prob_lr = lr.predict_proba(X_test)[:, 1]
y_prob_lr[:5]

array([0.65253835, 0.52739836, 0.0478591 , 0.04539213, 0.09992695])

In [11]:
def risk_level(prob):
    if prob < 0.3:
        return "Low Risk"
    elif prob < 0.6:
        return "Medium Risk"
    else:
        return "High Risk"


In [12]:
risk_labels = [risk_level(p) for p in y_prob_lr]
risk_labels[:10]

['High Risk',
 'Medium Risk',
 'Low Risk',
 'Low Risk',
 'Low Risk',
 'High Risk',
 'Medium Risk',
 'Low Risk',
 'Low Risk',
 'Low Risk']

Predicted probabilities from Logistic Regression were converted into Low, Medium, and High risk categories using predefined thresholds for better interpretability.]

In [13]:
import joblib

In [14]:
joblib.dump(lr, "../models/logistic_regression_model.pkl")
joblib.dump(scaler, "../models/scaler.pkl")

['../models/scaler.pkl']

In [15]:
import os
os.listdir("../models")

['logistic_regression_model.pkl', 'scaler.pkl']

In [16]:
type(lr)

sklearn.linear_model._logistic.LogisticRegression

In [17]:
import os
os.listdir("../models")

['logistic_regression_model.pkl', 'scaler.pkl']